# C6 lab: assemble v0 end to end, trace one query, and say which stage failed

Lesson: [`concepts/C6_v0_pipeline.md`](../concepts/C6_v0_pipeline.md), Part H. Concept card: [02 C6](../02_RAG_MASTER_LEARNING_PLAN.md#c6). Spec: [06 §2–§3](../06_RAG_PROJECT_AND_EVALUATION_PLAN.md#3-v0-baseline-exact-specification).

**How this lab works (blank-file first).**
- Each task has a **task cell**, an **empty code cell** for you, and a **CHECK cell** (`# CHECK`, do not edit). Run the CHECK after your code. If it fails, read the message and fix your code.
- This notebook contains **no solution code**. **FT: no.** v0 must be *your own* code: bring the functions you wrote in the C1-C5 labs (paste them, or import them from your `ragkit/` files). The names below are the names those labs used. `sources/verification/C6/` holds the author's reference run: it is a solution, so open it only after an honest attempt plus a hint from your tutor.
- **Predict first.** Some tasks ask for a number before you compute it. Fill it in honestly; it is part of the evidence.
- **No API calls by default.** The lab embeds with the **local MiniLM fallback** and answers with a labelled **fake client** (`FAKE_CLIENT`, shaped like C5's `FakeGroq`). `RUN_COHERE` and `RUN_GROQ` (Task 12) switch to the real providers (D-03, D-04). Never print key values.
- **Where files go.** The index goes to `index/<snapshot_id>/<params_hash>/<embed_hash>/` (06 §2.4 interface decisions; git-ignored: `chunks.jsonl` holds corpus text) and the run to `experiments/E000_v0_baseline/`, both under `rag_learning/`.
- Task 9 (toy diagnosis) needs no earlier task. You can start there if you like.

## Setup (given)
Run this cell. It sets paths and flags; `PARAMS` (06 §2.4's `chunking:` block, the course's `params_hash` input); the v0 template and helpers exactly as the C5 lab gave them (`ANSWER_V1_TXT`, `split_template`, `split_sentences`, `n_tok`, `sha256_hex`, `ABSTAIN`); `FAKE_CLIENT`, a labelled stand-in for `groq.Groq`; the ten questions (the nine draft seeds of 06 §6.7 and the repo's own test question); `_fp` for the CHECK cells; `_get` (so records may be dicts **or** objects); and the **TOY** traces for Task 9.

In [ ]:
import dataclasses, hashlib, json, os, re, subprocess, sys, time
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pymupdf, tiktoken

WS  = Path(r"C:\Yash\New-Work\Book RAG\rag_learning")
PDF = Path(r"C:\Yash\New-Work\Book RAG\upstream\RAG_Techniques\data\Understanding_Climate_Change.pdf")
OUT_ROOT = Path(os.environ.get("C6_LAB_OUT_ROOT", str(WS)))   # leave as is: index/ and experiments/ go here
RUN_COHERE = False     # Task 12: default embedder (D-03). Off: the local MiniLM fallback
RUN_GROQ = False       # Task 12: real generation (D-04). Off: FAKE_CLIENT answers (labelled toy)
RUN_CORPUS_T = False   # Task 12: only after Corpus T is chosen and frozen (DEC-3)

DOC_ID = "climate"
SNAPSHOT_ID_D = "ea07572562ac"     # 06 §5.3, Corpus D with PyMuPDF 1.26.7, no cleaning
PARAMS = {"name": "recursive_char", "unit": "chars", "chunk_size": 1000, "chunk_overlap": 200,
          "separators": ["\n\n", "\n", " ", ""], "scope": "per_page"}    # 06 §2.4 `chunking:` block (C2 lab PARAMS)
PARAMS_HASH_D = "1ca9c5a3b548"     # sha256(json.dumps(PARAMS, sort_keys=True))[:12], the course decision
K = 4                              # 06 §3.1
ENC = tiktoken.get_encoding("o200k_harmony")

def n_tok(text):
    return len(ENC.encode(text, disallowed_special=()))

def sha256_hex(s):
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

ANSWER_V1_TXT = (  # prompts/answer_v1.txt (06 §3.3), exactly as in the C5 lab
    "SYSTEM:\n"
    "You answer questions using ONLY the numbered sources below.\n"
    "Rules:\n"
    "1. Every factual sentence must end with one or more citations in the form [doc_id:page],\n"
    "   copied exactly from the source headers, e.g. [climate:10].\n"
    "2. If the sources do not contain the answer, reply exactly:\n"
    "   \"I cannot answer this from the provided documents.\"\n"
    "   Do not use outside knowledge.\n"
    "3. Do not cite a source that does not support the sentence.\n"
    "4. Be concise: at most 5 sentences.\n"
    "\n"
    "USER:\n"
    "Sources:\n"
    "{sources}\n"
    "\n"
    "Question: {question}\n"
    "Answer:\n")
ABSTAIN = "I cannot answer this from the provided documents."

def split_template(text):
    """'SYSTEM:\n<system>\nUSER:\n<user pattern>' -> (system, user_pattern). Given (C5)."""
    head, user = text.split("USER:\n", 1)
    assert head.startswith("SYSTEM:\n") and head.endswith("\n\n")
    return head[len("SYSTEM:\n"):-1], user.rstrip("\n")

def split_sentences(text):
    """Heuristic sentence splitter. Given (C5)."""
    t = re.sub(r"([.!?])((?:\s*\[[^\[\]\n]*\])+)", r"\2\1", text.strip())
    return [s for s in re.split(r"(?<=[.!?])\s+", t) if s.strip()]

SYSTEM_V0 = split_template(ANSWER_V1_TXT)[0]

class FakeClient:
    """FAKE stand-in for groq.Groq (labelled toy; same response shape as C5's FakeGroq). It answers by repeating the
    start of source [1] and citing that source's header. NOT a model; token counts are None (unknown, never invented)."""
    def __init__(self):
        self.calls = []
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))
    def _create(self, **kw):
        self.calls.append(kw)
        m = re.search(r"^\[1\] \(([^)]+)\)\n(.*)$", kw["messages"][-1]["content"], re.M)
        text = f"TOY ANSWER from FAKE_CLIENT: {m.group(2)[:60].strip()} [{m.group(1)}]."
        return SimpleNamespace(
            choices=[SimpleNamespace(index=0, finish_reason="stop", message=SimpleNamespace(role="assistant", content=text, reasoning=None))],
            usage=SimpleNamespace(prompt_tokens=None, completion_tokens=None, completion_tokens_details=None))

FAKE_CLIENT = FakeClient()
FAKE_MODEL_ID = "fake-client-v1"

SEEDS = [  # (qid, question): 06 §6.7 drafts D-001..D-009 + the repo's test query (simple_rag.ipynb cell 15)
    ("D-001", "According to the document, by how much have global temperatures risen since the late 19th century?"),
    ("D-002", "In what year was the Paris Agreement adopted?"),
    ("D-003", "According to the document, what is essential regarding countries' nationally determined contributions (NDCs) for meeting the Paris Agreement's goals?"),
    ("D-004", "What does DAC stand for, and what does it do?"),
    ("D-005", "How does the document contrast coal and natural gas in terms of carbon intensity?"),
    ("D-006", "What are the advantages of putting solar panels on lakes or reservoirs?"),
    ("D-007", "What is the supreme decision-making body of the convention that provides the framework for negotiating the Kyoto Protocol and the Paris Agreement?"),
    ("D-008", "What does the Kigali Amendment target?"),
    ("D-009", "In what year was the IPCC established?"),
    ("R-001", "What is the main cause of climate change?"),
]
Q = dict(SEEDS)
GOLD_PAGES = {"D-001": [3], "D-002": [10], "D-003": [23], "D-004": [8, 23], "D-005": [2], "D-006": [8],
              "D-007": [10, 29], "D-008": [24], "D-009": [], "R-001": None}   # 06 §6.7 drafts; None = no label

def _fp(x):
    """Fingerprint of a value. The CHECK cells compare fingerprints, not visible answers."""
    return hashlib.sha256(json.dumps(x, sort_keys=True).encode()).hexdigest()[:16]

def _get(obj, name):
    """Read a field from a dict or from an object (dataclass)."""
    return obj[name] if isinstance(obj, dict) else getattr(obj, name)

def _norm(s):
    return " ".join(s.split())

TOY_FILE = WS / "sources" / "verification" / "C6" / "toy_traces.json"
assert hashlib.sha256(TOY_FILE.read_bytes()).hexdigest()[:16] == "a6d8cb1b44d7b829", "toy_traces.json changed"
TOY = json.loads(TOY_FILE.read_text(encoding="utf-8"))
print("OUT_ROOT:", OUT_ROOT)
print("TOY:", TOY["label"][:80], "...", len(TOY["traces"]), "traces")

## Task 1: draw v0 from memory (order the steps)

Close the lesson. Put the eleven steps of `STEP_BANK` into `MY_ORDER`, in the order v0 runs them for one corpus and then one question. Copy the strings exactly.

Tip: the first five run once per corpus (index time); the rest run once per question (query time).

In [ ]:
STEP_BANK = [   # shuffled; copy each string once into MY_ORDER
    "extract and resolve the citations",
    "embed the chunks",
    "build the prompt (rules + numbered blocks + question)",
    "load pages (PyMuPDF, one text per page)",
    "append the trace record to results.jsonl",
    "score every chunk and take the top-k",
    "write the index (vectors.npy, chunks.jsonl, manifest.json)",
    "freeze the text snapshot (snapshot_id)",
    "embed the question",
    "generate the answer",
    "chunk (offsets, page_segments, chunk_id)",
]
MY_ORDER = [
]

In [ ]:
# CHECK Task 1 (do not edit)
assert sorted(MY_ORDER) == sorted(STEP_BANK), "use every string of STEP_BANK exactly once"
assert MY_ORDER.index("embed the question") > MY_ORDER.index("write the index (vectors.npy, chunks.jsonl, manifest.json)"), \
    "the question is embedded at query time, after the index exists"
assert _fp(MY_ORDER) == "bf21ce01fbde3748", \
    "the order is not right yet. For each step ask: what must already exist before it can run?"
print("Task 1 OK")

## Task 2: load and freeze the snapshot (your C1 code)

Bring your C1 functions (C1 lab Tasks 1-3):
- `load_pages(pdf_path, doc_id)` → a list of pages, each with `page` (1-based), `text` (exactly `page.get_text()`, **no cleaning**) and `parser`;
- `build_page_table(page_texts)` → `(doc_text, rows)`, rows `{"page", "doc_char_start", "doc_char_end", "text_sha256"}`;
- `snapshot_id(backend, version, options, cleaning_version, doc_text_sha256)` with the **exact** 06 §5.3 formula.

Then set `pages = load_pages(PDF, DOC_ID)`, `page_texts = [_get(p, "text") for p in pages]`, `doc_text, page_rows = build_page_table(page_texts)` and `SID = snapshot_id("pymupdf", pymupdf.VersionBind, {}, "none", {DOC_ID: sha256_hex(doc_text)})`.

In [ ]:
# Your code here (blank-file first: your own C1-C5 code; no copying from sources/verification/C6/).

In [ ]:
# CHECK Task 2 (do not edit)
assert len(pages) == 33 and [_get(p, "page") for p in pages] == list(range(1, 34)), "33 pages, numbered from 1"
assert page_texts == [p.get_text() for p in pymupdf.open(PDF)], "v0 does no cleaning: each text must be exactly page.get_text()"
assert doc_text == "\n".join(page_texts), "doc_text = the pages joined with '\\n'"
assert page_rows[0]["doc_char_start"] == 0 and all(b["doc_char_start"] == a["doc_char_end"] + 1 for a, b in zip(page_rows, page_rows[1:]))
assert all(doc_text[r["doc_char_start"]:r["doc_char_end"]] == t for r, t in zip(page_rows, page_texts)), "each row must slice out its page"
assert SID == SNAPSHOT_ID_D, f"snapshot_id {SID} != {SNAPSHOT_ID_D}: check the 06 §5.3 formula (page_join is a literal backslash-n)"
print(f"Task 2 OK: {len(pages)} pages, {len(doc_text):,} characters, snapshot_id {SID}")

## Task 3: chunk, with your own chunker (C2; Gate 2→3 item 3)

Bring the chunker you **rewrote from memory** in C2 (C2 lab Tasks 5-7; it must pass your round-trip checks):
- `params_hash(params)` → the first 12 hex of `sha256(json.dumps(params, sort_keys=True).encode("utf-8"))`;
- `make_records(doc_id, snapshot_id, page_texts, params)` → one dict per chunk, keys in 06 §2.3 order: `doc_id, snapshot_id, chunk_id, seq, char_start, char_end, page_segments, text, text_sha256, chunker, params_hash, meta`; `chunk_id = f"{doc_id}:{snapshot_id}:{params_hash}:{seq:05d}"`; per page, recursive, 1,000 / 200 characters (`PARAMS`).

Then set `chunks = make_records(DOC_ID, SID, page_texts, PARAMS)`.

**Predict first:** how many chunks will v0 make on the climate PDF? (33 pages of about 2,200 characters each.)

In [ ]:
my_prediction_n_chunks = None    # BEFORE you run your chunker

In [ ]:
# Your code here (blank-file first: your own C1-C5 code; no copying from sources/verification/C6/).

In [ ]:
# CHECK Task 3 (do not edit)
assert my_prediction_n_chunks is not None, "write your prediction first"
_ph = params_hash(PARAMS)
assert _ph == PARAMS_HASH_D, f"params_hash(PARAMS) = {_ph}; the course value is {PARAMS_HASH_D} (sha256 of json.dumps(PARAMS, sort_keys=True))"
assert params_hash({**PARAMS, "chunk_overlap": 100}) != _ph, "a different overlap must give a different params_hash"
_starts = [r["doc_char_start"] for r in page_rows]
for i, c in enumerate(chunks):
    t, a, b = _get(c, "text"), _get(c, "char_start"), _get(c, "char_end")
    assert _get(c, "seq") == i, "seq runs 0, 1, 2, ... within a document (Corpus D has one document; seq restarts per document)"
    assert doc_text[a:b] == t, f"chunk {i}: doc_text[char_start:char_end] != text (offsets are DOCUMENT offsets)"
    assert 0 < len(t) <= PARAMS["chunk_size"], f"chunk {i} has length {len(t)}"
    assert _get(c, "chunk_id") == f"{DOC_ID}:{SID}:{_ph}:{i:05d}", f"chunk {i}: chunk_id must be doc_id:snapshot_id:params_hash:seq"
    assert _get(c, "params_hash") == _ph and _get(c, "snapshot_id") == SID
    assert _get(c, "text_sha256") == sha256_hex(t)
    segs = _get(c, "page_segments")
    assert len(segs) == 1, f"chunk {i}: per-page chunks have exactly one page segment"
    p, sa, sb = _get(segs[0], "page"), _get(segs[0], "char_start"), _get(segs[0], "char_end")
    assert (_starts[p - 1] + sa, _starts[p - 1] + sb) == (a, b), f"chunk {i}: the page segment must decode back to [char_start, char_end)"
assert [_get(c, "chunk_id") for c in make_records(DOC_ID, SID, page_texts, PARAMS)] == [_get(c, "chunk_id") for c in chunks], \
    "rebuilding must give identical chunk ids"
_same_as_ref = _fp([_get(c, "chunk_id") for c in chunks]) == "afdaa756486c15d4"
print(f"Task 3 OK: {len(chunks)} chunks (you predicted {my_prediction_n_chunks}); params_hash {_ph}")
print("Same chunk ids as the C2 lab / C6 reference (97 chunks):", _same_as_ref,
      "" if _same_as_ref else "(fine if your round-trip tests pass; write down WHY your count differs)")

## Task 4: embed and write the index (C3 + C4's embedder)

Bring your code:
- `embedder` (C4 lab): an object with `model_id`, `dim`, `embed_documents(texts)` → `(n, dim)` and `embed_query(text)` → `(dim,)`, both **L2-normalised float32**. Default: the local fallback `sentence-transformers/all-MiniLM-L6-v2` (revision `1110a243fdf4706b3f48f1d95db1a4f5529b4d41`, 06 §2.4). Task 12 switches to Cohere;
- `V`: the chunk vectors, from your C3 `cached_embed` or `embedder.embed_documents`. **Row i of `V` = line i of `chunks.jsonl`** (06 §2.4). Never assume `seq == i`: `seq` restarts in every document (Corpus T has several);
- `embed_hash(embedding)` (C3): the first 12 hex of `sha256(json.dumps({provider, model_id, revision, input_type, normalize, preproc_version}, sort_keys=True).encode("utf-8"))` (06 §2.4);
- `MANIFEST` (C3 lab Task 4a): at least `snapshot_id`, `docs`, `parser`, `chunker` (= `PARAMS`), `params_hash`, `embedding` (`provider`, `model_id`, `revision`, `input_type`, `preproc_version`, `normalize`, `dim`) and `embed_hash`;
- `write_index(index_dir, chunks, V, manifest)` → the written manifest, which adds `files` (`vectors.npy`: `sha256`, `shape`; `chunks.jsonl`: `sha256`, `lines`) (C3 lab Task 4);
- `load_index(index_dir, run_cfg)` → `(V, chunks, manifest)`, raising `ValueError` on any mismatch (C3 lab Tasks 5-6). `RUN` is your v0 run config: at least `snapshot_id`, `chunker`, `embedding`.

Then set `INDEX_DIR = OUT_ROOT / "index" / SID / params_hash(PARAMS) / embed_hash(MANIFEST["embedding"])`, `written = write_index(INDEX_DIR, chunks, V, MANIFEST)` and `V_l, chunks_l, manifest_l = load_index(INDEX_DIR, RUN)`.

**Predict first:** how many bytes will `vectors.npy` take? (float32 is 4 bytes; `.npy` adds a 128-byte header.)

In [ ]:
my_prediction_vectors_bytes = None    # BEFORE you build the index

In [ ]:
# Your code here (blank-file first: your own C1-C5 code; no copying from sources/verification/C6/).

In [ ]:
# CHECK Task 4 (do not edit)
assert my_prediction_vectors_bytes is not None, "write your prediction first"
_EF = ("provider", "model_id", "revision", "input_type", "normalize", "preproc_version")
_eh = hashlib.sha256(json.dumps({k: MANIFEST["embedding"][k] for k in _EF}, sort_keys=True).encode("utf-8")).hexdigest()[:12]
assert embed_hash(MANIFEST["embedding"]) == _eh, "embed_hash: 12 hex of sha256(json.dumps(<the six embedding fields>, sort_keys=True)) (06 §2.4)"
assert INDEX_DIR == OUT_ROOT / "index" / SID / params_hash(PARAMS) / _eh, "INDEX_DIR = OUT_ROOT / 'index' / SID / params_hash(PARAMS) / embed_hash(...)"
assert sorted(p.name for p in INDEX_DIR.iterdir()) == ["chunks.jsonl", "manifest.json", "vectors.npy"], "exactly three files"
_m = json.loads((INDEX_DIR / "manifest.json").read_text(encoding="utf-8"))
for key in ("snapshot_id", "chunker", "params_hash", "embedding", "embed_hash", "files"):
    assert key in _m, f"manifest.json lacks {key!r}"
assert _m["snapshot_id"] == SID and _m["params_hash"] == params_hash(PARAMS) and _m["chunker"] == PARAMS and _m["embed_hash"] == _eh
for key in ("model_id", "revision", "normalize", "dim"):
    assert key in _m["embedding"], f"manifest['embedding'] lacks {key!r}"
assert _m["files"]["vectors.npy"]["sha256"] == hashlib.sha256((INDEX_DIR / "vectors.npy").read_bytes()).hexdigest()
assert _m["files"]["chunks.jsonl"]["sha256"] == hashlib.sha256((INDEX_DIR / "chunks.jsonl").read_bytes()).hexdigest()
assert isinstance(V_l, np.ndarray) and V_l.dtype == np.float32 and V_l.shape == (len(chunks), _m["embedding"]["dim"])
assert np.allclose(np.linalg.norm(V_l, axis=1), 1.0, atol=1e-5), "rows of V must be L2-normalised"
assert [_get(c, "chunk_id") for c in chunks_l] == [_get(c, "chunk_id") for c in chunks], "line i of chunks.jsonl must be chunk i (= row i of V)"
try:
    load_index(INDEX_DIR, {**RUN, "chunker": {**PARAMS, "chunk_overlap": 100}})
    raise AssertionError("load_index accepted a run config with another chunker; it must raise ValueError")
except ValueError:
    pass
_vb = (INDEX_DIR / "vectors.npy").stat().st_size
print(f"Task 4 OK: V {V_l.shape}; vectors.npy {_vb:,} bytes (you predicted {my_prediction_vectors_bytes}); embed_hash {_eh}")
print("Same embedding block as C3 / the C6 reference (embed_hash c398eeba043f):", _eh == "c398eeba043f")

## Task 5: retrieve (C4)

Bring your C4 function `retrieve(question, embedder, V, chunks, k=4)` (C4 lab Task 2): exact cosine (`V @ q`), the top `k`, each item `{"rank", "chunk_id", "doc_id", "page", "char_start", "char_end", "score", "text"}`, `rank` from 1, `page` = the first page segment's page, **equal scores broken by `chunk_id`** (06 §3.1).

Then set `retrieved = retrieve(Q["D-002"], embedder, V_l, chunks_l, K)`.

**Predict first:** the D-002 answer ("2015") is on page 10. At which rank will page 10 appear, if at all?

In [ ]:
my_prediction_rank_p10 = None    # 1, 2, 3, 4 or 'not in top 4'

In [ ]:
# Your code here (blank-file first: your own C1-C5 code; no copying from sources/verification/C6/).

In [ ]:
# CHECK Task 5 (do not edit)
assert my_prediction_rank_p10 is not None, "write your prediction first"
class _ToyEmb:   # toy: a fixed query vector
    def embed_query(self, text): return np.array([1, 0], dtype=np.float32)
_toy_chunks = [{"chunk_id": cid, "doc_id": "toy", "seq": i, "char_start": 0, "char_end": 1, "text": "x",
                "page_segments": [{"page": 1, "char_start": 0, "char_end": 1}]} for i, cid in enumerate(("toy:b", "toy:a", "toy:c", "toy:d"))]
_toy_V = np.array([[1, 0], [1, 0], [0, 1], [0.6, 0.8]], dtype=np.float32)
_r = retrieve("toy", _ToyEmb(), _toy_V, _toy_chunks, 3)
assert [x["chunk_id"] for x in _r] == ["toy:a", "toy:b", "toy:d"], f"tie rule: equal scores sorted by chunk_id; got {[x['chunk_id'] for x in _r]}"
assert [x["rank"] for x in _r] == [1, 2, 3]
assert len(retrieved) == K and [c["rank"] for c in retrieved] == [1, 2, 3, 4]
assert all(a["score"] >= b["score"] for a, b in zip(retrieved, retrieved[1:])), "scores must be in descending order"
_row = {_get(c, "chunk_id"): i for i, c in enumerate(chunks_l)}
_qv = embedder.embed_query(Q["D-002"])
for c in retrieved:
    assert abs(float(V_l[_row[c["chunk_id"]]] @ _qv) - c["score"]) < 1e-5, "the stored score must equal V[row] @ q"
    assert doc_text[c["char_start"]:c["char_end"]] == c["text"], f"rank {c['rank']}: doc_text[char_start:char_end] must equal its text"
if globals().get("_same_as_ref") and "MiniLM-L6" in str(getattr(embedder, "model_id", "")):
    assert _fp([c["chunk_id"] for c in retrieved]) == "78ebfed94d0021da", "same chunks and model as the reference, so the top 4 must match it"
print("Task 5 OK: top-4 pages", [c["page"] for c in retrieved], "scores", [round(c["score"], 4) for c in retrieved],
      f"| you predicted {my_prediction_rank_p10}")

**Look at the scores of ranks 3 and 4.** What does their difference tell you about how stable this top-4 is?

*Template:* "The gap is about ____, so a small change in ____ could ____."

*Your answer:*

## Task 6: prompt, answer and citations (C5)

Bring your C5 functions (C5 lab Tasks 1-6): `build_context`, `build_messages(question, retrieved, template_text=ANSWER_V1_TXT)`, `call_llm(client, messages, *, model_id, temperature=0.0, max_tokens=512, reasoning_effort="low")`, `parse_citations`, `check_citations`, `is_abstention`, `make_answer`, `make_record(qid, question, retrieved, messages, llm_result, template_text=ANSWER_V1_TXT)`.

Then set `messages = build_messages(Q["D-002"], retrieved)`, `llm_result = call_llm(FAKE_CLIENT, messages, model_id=FAKE_MODEL_ID)` and `record = make_record("D-002", Q["D-002"], retrieved, messages, llm_result)`.

In [ ]:
# Your code here (blank-file first: your own C1-C5 code; no copying from sources/verification/C6/).

In [ ]:
# CHECK Task 6 (do not edit)
assert messages[0] == {"role": "system", "content": SYSTEM_V0}, "the system message must be the v0 rules"
_u = messages[1]["content"]
_h = re.findall(r"^\[(\d+)\] \(([^:()]+):(\d+)\)$", _u, re.M)
assert [int(i) for i, _, _ in _h] == [1, 2, 3, 4] and [int(p) for _, _, p in _h] == [c["page"] for c in retrieved], \
    "headers '[i] (doc_id:page)' numbered 1..4 in rank order"
assert _u.startswith("Sources:\n") and _u.endswith(f"Question: {Q['D-002']}\nAnswer:")
assert llm_result["text"].startswith("TOY ANSWER") or RUN_GROQ
assert record["prompt"]["sha256"] == sha256_hex(json.dumps(messages, sort_keys=True, ensure_ascii=False)), "prompt.sha256 (C5 recipe)"
assert record["context"]["chunk_ids"] == [c["chunk_id"] for c in retrieved]
_c = record["answer"]["citations"]
assert _c and _c[0]["resolved_chunk_ids"] == [retrieved[0]["chunk_id"]], "the fake answer cites source [1], so it resolves to rank 1"
print("Task 6 OK. Answer:", llm_result["text"][:100])
print("Citations:", _c)

## Task 7: the trace record and `run_v0` (D1 starts here)

Write two functions:
- `make_trace(...)` (your signature): one 06 §2.5 record per question. Take C5's `record` and add the C6 envelope: `trace_version`, `run_id`, `qid`, `timestamp_utc`, `config_hash`, `index_manifest_hash`, `snapshot_id`, `query` (`original`, `transformed`), `candidates` (`dense`: C4's list **without** `text`), `latency_ms` (`embed_query` or `null`, `dense`, `context`, `llm`, `total`), `cost_usd_estimate`, `errors`. With the fake client, set `llm.provider = "fake"` and `llm.model_id = FAKE_MODEL_ID`: a fake answer must never be labelled `groq`. Use `None` for anything you cannot know (for example tokens from the fake client); never invent a value;
- `run_v0(questions, run_dir) -> Path`: for a list of `(qid, question)`, write `config.json` (your v0 config; `config_hash` = `"sha256:"` + the sha256 of that file's **bytes**), then write one trace per question to a fresh `run_dir / "results.jsonl"`. `index_manifest_hash` = `"sha256:"` + the sha256 of `manifest.json`'s bytes.

Then set `RUN_DIR = OUT_ROOT / "experiments" / "E000_v0_baseline"` and `RESULTS = run_v0(SEEDS, RUN_DIR)`.

**Predict first:** how many bytes is one trace record (one JSON line)? It stores ids, offsets and scores, not chunk texts or vectors.

In [ ]:
my_prediction_trace_bytes = None    # BEFORE you run run_v0

In [ ]:
# Your code here (blank-file first: your own C1-C5 code; no copying from sources/verification/C6/).

In [ ]:
# CHECK Task 7 (do not edit)
assert my_prediction_trace_bytes is not None, "write your prediction first"
_lines = Path(RESULTS).read_text(encoding="utf-8").splitlines()
assert len(_lines) == len(SEEDS), f"results.jsonl must hold one record per question ({len(SEEDS)}), found {len(_lines)}"
TRACES = [json.loads(l) for l in _lines]
REQ = {"trace_version", "run_id", "qid", "timestamp_utc", "config_hash", "index_manifest_hash", "snapshot_id", "query",
       "candidates", "context", "prompt", "answer", "latency_ms", "cost_usd_estimate", "llm", "errors"}
_ids = {_get(c, "chunk_id"): c for c in chunks_l}
_cfg_bytes = (Path(RUN_DIR) / "config.json").read_bytes()
for t in TRACES:
    miss = REQ - set(t)
    assert not miss, f"{t.get('qid')}: missing fields {sorted(miss)}"
    for sub, keys in (("query", ["original", "transformed"]), ("context", ["chunk_ids", "context_tokens", "tokenizer", "truncated_chunks"]),
                      ("prompt", ["template_version", "sha256", "input_tokens"]), ("answer", ["text", "abstained", "citations", "output_tokens"]),
                      ("latency_ms", ["embed_query", "dense", "context", "llm", "total"]), ("llm", ["provider", "model_id", "cache_hit", "temperature"])):
        assert all(k in t[sub] for k in keys), f"{t['qid']}: {sub} needs {keys}"
    assert t["snapshot_id"] == SID
    assert t["config_hash"] == "sha256:" + hashlib.sha256(_cfg_bytes).hexdigest(), "config_hash = 'sha256:' + sha256(config.json bytes)"
    assert t["index_manifest_hash"] == "sha256:" + hashlib.sha256((INDEX_DIR / "manifest.json").read_bytes()).hexdigest()
    dense = t["candidates"]["dense"]
    assert [c["rank"] for c in dense] == list(range(1, K + 1)) and all("text" not in c for c in dense), "candidates: ranks 1..k, no chunk text"
    for c in dense:
        assert c["chunk_id"] in _ids, f"{t['qid']}: candidate {c['chunk_id']} is not in chunks.jsonl"
        assert doc_text[c["char_start"]:c["char_end"]] == _get(_ids[c["chunk_id"]], "text"), "candidate offsets must reproduce the chunk text"
    assert t["context"]["chunk_ids"] == [c["chunk_id"] for c in dense], "v0 context = the top-k in rank order"
    _ret = [{**c, "text": _get(_ids[c["chunk_id"]], "text")} for c in dense]
    _msgs = build_messages(t["query"]["original"], _ret)
    assert t["prompt"]["sha256"] == sha256_hex(json.dumps(_msgs, sort_keys=True, ensure_ascii=False)), f"{t['qid']}: prompt.sha256"
    for x in t["answer"]["citations"]:
        assert set(x["resolved_chunk_ids"]) <= set(t["context"]["chunk_ids"])
    assert isinstance(t["answer"]["abstained"], bool) and isinstance(t["errors"], list)
    if not RUN_GROQ:
        assert t["llm"]["provider"] == "fake" and t["llm"]["model_id"] == FAKE_MODEL_ID, "a fake answer must be labelled fake"
_sizes = [len(l.encode("utf-8")) + 1 for l in _lines]
print(f"Task 7 OK: {len(TRACES)} traces; bytes per record min/max {min(_sizes)}/{max(_sizes)} (you predicted {my_prediction_trace_bytes})")

### Task 7b: rebuild from scratch (Gate 2→3 item 4)
The next CHECK rebuilds the pages, the snapshot id, the chunks, the vectors and the index into a **fresh** folder with your own functions, then compares them with the first build. Nothing to write; just run it. If it fails, find what is not deterministic (an unsorted set? a different embedding batch? a clock value?). A `created_utc` field in the manifest is allowed to differ; nothing else is.

In [ ]:
# CHECK Task 7b (do not edit)
_pt2 = [_get(p, "text") for p in load_pages(PDF, DOC_ID)]
_dt2, _ = build_page_table(_pt2)
_sid2 = snapshot_id("pymupdf", pymupdf.VersionBind, {}, "none", {DOC_ID: sha256_hex(_dt2)})
_chunks2 = make_records(DOC_ID, _sid2, _pt2, PARAMS)
_V2 = embedder.embed_documents([_get(c, "text") for c in _chunks2])
_dir2 = OUT_ROOT / ".cache" / "c6_rebuild_check" / _sid2 / params_hash(PARAMS) / embed_hash(MANIFEST["embedding"])
import shutil; shutil.rmtree(_dir2, ignore_errors=True)
write_index(_dir2, _chunks2, _V2, MANIFEST)
assert _sid2 == SID and params_hash(PARAMS) == _m["params_hash"], "snapshot_id or params_hash changed on rebuild"
assert [_get(c, "chunk_id") for c in _chunks2] == [_get(c, "chunk_id") for c in chunks], "chunk ids changed on rebuild"
for f in ("chunks.jsonl", "vectors.npy"):
    assert (INDEX_DIR / f).read_bytes() == (_dir2 / f).read_bytes(), f"{f} is not byte-identical after a rebuild"
_drop = lambda m: {k: v for k, v in m.items() if k != "created_utc"}
assert _drop(json.loads((INDEX_DIR / "manifest.json").read_text(encoding="utf-8"))) == _drop(json.loads((_dir2 / "manifest.json").read_text(encoding="utf-8"))), \
    "manifest.json differs after a rebuild (other than created_utc)"
print(f"Task 7b OK: snapshot {SID}, params_hash {params_hash(PARAMS)}, {len(_chunks2)} identical chunk ids; chunks.jsonl and vectors.npy byte-identical")

## Task 8: walk one query through every artifact (Gate 2→3 item 2)

Write `walk(trace)` for the D-002 record. For each context chunk, in rank order, it **prints** the chain *page text → chunk + offsets → vector → score → rank*, and **returns** a list of dicts
`{"rank", "chunk_id", "row", "page", "page_start", "page_end", "doc_start", "doc_end", "score_stored", "score_recomputed"}`, where
- `row` is the chunk's line in `chunks.jsonl` (= its row in `V_l`);
- `page_start, page_end` come from its page segment, and `doc_start, doc_end` are those converted to **document** offsets with your page table;
- `score_recomputed` is `V_l[row] @ embedder.embed_query(question)`.

After the loop, print the prompt hash, the answer and its resolved citations.

In [ ]:
# Your code here (blank-file first: your own C1-C5 code; no copying from sources/verification/C6/).

In [ ]:
# CHECK Task 8 (do not edit)
_t = next(t for t in TRACES if t["qid"] == "D-002")
_w = walk(_t)
assert [x["rank"] for x in _w] == [1, 2, 3, 4] and [x["chunk_id"] for x in _w] == _t["context"]["chunk_ids"]
for x in _w:
    c = chunks_l[x["row"]]
    assert _get(c, "chunk_id") == x["chunk_id"], "row must point at the chunk's line in chunks.jsonl"
    assert (x["doc_start"], x["doc_end"]) == (_get(c, "char_start"), _get(c, "char_end")), "page segment -> document offsets must give [char_start, char_end)"
    assert page_texts[x["page"] - 1][x["page_start"]:x["page_end"]] == _get(c, "text"), "the page slice must be the chunk text"
    assert abs(x["score_recomputed"] - x["score_stored"]) < 1e-5, "recomputed score must equal the stored score"
print("Task 8 OK")

**Explain each artifact in writing** (Gate 2→3 item 2). One line each, in your own words, using the D-002 values you printed.

*Template:* "page text: ____ · chunk + offsets: ____ · vector: ____ · scores: ____ · top-k: ____ · prompt: ____ · answer: ____ · citations: ____."

*Your answer:*

## Task 9: which stage failed? (TOY traces; CR-11)

`TOY` holds seven **made-up** traces over a made-up 5-page manual (not a real run). Each has the parsed `pages`, the full chunk list, the question, a **gold quote** (what a human reads on the page), the context chunks and the answer. Some contain a planted failure.

Labels (pick the **first** stage that broke, walking upstream → downstream):
`"parse"` · `"chunk"` · `"retrieve"` · `"generate:citation"` · `"generate:abstain"` · `"generate:grounding"` · `"ok"`.

**9a. By hand first.** Run the next cell to print the traces, then fill `MY_LABELS`.

In [ ]:
_by = {c["chunk_id"]: c for c in TOY["chunks"]}
for t in TOY["traces"]:
    print(f"--- {t['qid']}  Q: {t['query']['original']}")
    print(f"    gold p.{t['gold']['page']}: {t['gold']['quote']!r}")
    print(f"    parsed page {t['gold']['page']}: {TOY['pages'][t['gold']['page'] - 1]!r}")
    for i in t["context"]["chunk_ids"]:
        print(f"    context: p.{_by[i]['page']} {_by[i]['text']!r}")
    print(f"    answer: {t['answer']['text']!r}")
print("all chunks:", [c["text"] for c in TOY["chunks"]])

In [ ]:
MY_LABELS = {"TOY-1": None, "TOY-2": None, "TOY-3": None, "TOY-4": None, "TOY-5": None, "TOY-6": None, "TOY-7": None}

In [ ]:
# CHECK Task 9a (do not edit)
_VOCAB = {"parse", "chunk", "retrieve", "generate:citation", "generate:abstain", "generate:grounding", "ok"}
assert all(v in _VOCAB for v in MY_LABELS.values()), f"every label must be one of {sorted(_VOCAB)}"
assert _fp(MY_LABELS) == "24494e44adb7c4d6", \
    "at least one label is not right yet. For each trace: is the quote in the parsed page? in some chunk? in the context? then look at the answer."
print("Task 9a OK")

**TOY-7 in one sentence.** The answer is correct and cites page 1, yet the label is not `"ok"` and not `"generate:citation"`. Why?

*Template:* "The first broken artifact is ____, because ____; the citation problem is a ____ of it."

*Your answer:*

**9b. Now as code.** Write `first_failing_stage(trace, pages, chunks) -> str` that applies these checks in order and returns the first that fails (compare texts after `_norm`, whitespace only):
1. `"parse"`: the gold quote is not in the parsed text of the gold page;
2. `"chunk"`: no chunk (in `chunks`) holds the whole quote;
3. `"retrieve"`: no **context** chunk holds the whole quote;
4. `"generate:citation"`: the answer cites a `(doc_id, page)` that no context chunk comes from;
5. `"generate:abstain"`: the answer abstained although the evidence is in the context;
6. `"generate:grounding"`: a number in the answer does not appear in the text of the context chunks it cites;
7. otherwise `"ok"`.

In [ ]:
# Your code here (blank-file first: your own C1-C5 code; no copying from sources/verification/C6/).

In [ ]:
# CHECK Task 9b (do not edit)
_got = {t["qid"]: first_failing_stage(t, TOY["pages"], TOY["chunks"]) for t in TOY["traces"]}
if _got.get("TOY-1") == "generate:grounding":
    raise AssertionError("TOY-1 came out 'generate:grounding': did you count the page number inside [toy:1] as a claim? Remove [doc:page] markers before collecting numbers.")
assert _got.get("TOY-7") == "retrieve", f"TOY-7 gave {_got.get('TOY-7')!r}: check the stages in upstream-to-downstream order"
assert _fp(_got) == "24494e44adb7c4d6", f"your labels {_got} differ from the planted ones"
_x = json.loads(json.dumps(TOY["traces"][0]))
_x["answer"] = {"text": ABSTAIN, "abstained": True, "citations": []}
assert first_failing_stage(_x, TOY["pages"], TOY["chunks"]) == "generate:abstain", "evidence in context + refusal sentence = generate:abstain"
_y = json.loads(json.dumps(TOY["traces"][5]))
_y["answer"]["text"] = "Each keeper works a 12-hour shift [toy:5]."
assert first_failing_stage(_y, TOY["pages"], TOY["chunks"]) == "ok", "TOY-6 with the right number is ok"
print("Task 9b OK:", _got)

## Task 10: ten manual queries and failure notes (Gate 2→3 item 5; seeds M5)

Your `results.jsonl` from Task 7 holds the ten questions. The next cell (given) prints one line per trace: the top-4 pages and scores, the rank of the first chunk on a gold page (06 §6.7 drafts; `None` = no label), and the answer with its `finish_reason` and flags. At the end it counts two flags over the whole run (C5 D.6): `length_stop` (the reply hit the token cap, `finish_reason` "length") and `unexpected_finish_reason` (anything other than "stop" or "length", or no value). A cut-off answer can look like a citation failure (C5-A-03), so read these counts before you blame the citations. With `RUN_GROQ = False` the answers are fake, so your notes can only cover retrieval. **For the gate run, switch on Groq (Task 12), rerun Task 7, and come back here.** Replace at least three of the ten questions with your own before the gate run.

In [ ]:
for t in TRACES:
    d = t["candidates"]["dense"]
    gp = GOLD_PAGES.get(t["qid"])
    first_gold = None if not gp else next((c["rank"] for c in d if c["page"] in gp), "not in top 4")
    print(f"{t['qid']}: pages {[c['page'] for c in d]} scores {[round(c['score'], 3) for c in d]} | gold pages {gp} -> first gold rank {first_gold}")
    print(f"        answer: {str(t['answer']['text'])[:110]}")
    print(f"        finish_reason {t['llm'].get('finish_reason')!r} | flags {t['answer'].get('flags')}")

# Run report: the two finish_reason flags (C5 D.6), counted over the whole run.
_FINISH_FLAGS = ("length_stop", "unexpected_finish_reason")
def _finish_flags(fr):   # "stop" -> none; "length" -> length_stop; anything else, or None -> unexpected_finish_reason
    return set() if fr == "stop" else {"length_stop"} if fr == "length" else {"unexpected_finish_reason"}
_n = {f: sum(f in (t["answer"].get("flags") or []) for t in TRACES) for f in _FINISH_FLAGS}
print(f"\nRun flags over {len(TRACES)} traces: length_stop {_n['length_stop']} | unexpected_finish_reason {_n['unexpected_finish_reason']}")
_odd = [t["qid"] for t in TRACES
        if set(t["answer"].get("flags") or []) & set(_FINISH_FLAGS) != _finish_flags(t["llm"].get("finish_reason"))]
if _odd:
    print("  These flags do not match llm.finish_reason for", _odd, "(check your C5 make_answer)")

Fill one row per question. This is **open coding** (method: AE Ch3, 'Open Coding: Reading and Labeling Traces'): read the whole trace as the person who asked, and write down what you see. Do not sort into categories yet.

| qid | Would a reader accept it? (Y/N; Groq run only) | What went wrong (first problem, in your own words) | First failing stage | Trace evidence (field = value) |
|---|---|---|---|---|
| | | | | |

- **Would a reader accept it?** Y or N, never "maybe": a forced choice makes you decide what counts as a failure. Leave it empty on the fake run; a fake answer has nothing to accept.
- **What went wrong.** Only the **first** problem, the most upstream one; later problems are often its knock-on effects. A short phrase, clear enough that someone else understands it without opening the trace. Write "nothing seen" when nothing went wrong.
- **First failing stage.** Use the Task 9 ladder.
- **A "Y" can still be a failure.** If the answer is right but the first failing stage is `retrieve`, the model answered from memory, not from your documents (06 §7.1, the danger zone; TOY-7). Count it as a failure.

**After all ten rows, group your notes.** Put notes about the same kind of problem together, and give each group a short name of your own. Your notes are your first **open codes** (raw failure notes, before any categories). These groups are a first draft: M5's method regroups and counts them later.

| Group name (yours) | qids | How many |
|---|---|---|
| | | |

## Task 11: one command, clean environment (Gate 2→3 item 1)

Move your functions into `ragkit/` files (06 §10: `loaders.py`, `chunking.py`, `index.py`, `retrieval.py`, `context.py`, `generate.py`, `providers/`, `tracing.py`) plus one entry point. Then set `V0_COMMAND` to the **one** command that runs v0 on Corpus D from `rag_learning/`, and `V0_RESULTS` to the `results.jsonl` it writes. The CHECK runs it in a fresh process (it uses no state from this notebook) and compares its chunk ids and retrieval with Task 7. For the gate, also run it once from a fresh clone after `uv sync --locked` (as in O2) and log it.

In [ ]:
V0_COMMAND = None    # e.g. a string such as  uv run python -m ragkit.run_v0 --corpus D --out <dir>
V0_RESULTS = None    # Path of the results.jsonl that command writes

In [ ]:
# CHECK Task 11 (do not edit)
assert V0_COMMAND and V0_RESULTS, "set V0_COMMAND and V0_RESULTS"
_p = subprocess.run(V0_COMMAND, shell=True, cwd=WS, capture_output=True, text=True, timeout=900)
assert _p.returncode == 0, f"the command failed (exit {_p.returncode}):\n{_p.stderr[-1500:]}"
_cli = [json.loads(l) for l in Path(V0_RESULTS).read_text(encoding="utf-8").splitlines()]
assert [t["qid"] for t in _cli] == [t["qid"] for t in TRACES], "run the same ten questions"
_strip = lambda t: [(c["chunk_id"], round(c["score"], 6)) for c in t["candidates"]["dense"]]
assert [_strip(t) for t in _cli] == [_strip(t) for t in TRACES], "the one-command run must retrieve exactly what the notebook retrieved"
print(f"Task 11 OK: one command, {len(_cli)} traces, same retrieval as Task 7")

## Task 12 (optional; spends API credit): real providers, and Corpus T

**Not run by the course author. Expected behaviour is UNVERIFIED.**
- **Groq (D-04).** Set `RUN_GROQ = True`, define `answer_fn` with your C5 Groq function (model id from `.env` `GROQ_MODEL_ID`, `temperature=0`, `reasoning_effort="low"`, enough `max_tokens`; log `usage` and `finish_reason`), rerun Tasks 6-7 and 10. Ten questions cost a few thousand tokens.
- **Cohere (D-03).** Set `RUN_COHERE = True` and use your C3/C4 Cohere embedder (model id from `.env` `COHERE_EMBED_MODEL_ID`; `input_type` `search_document` for chunks and `search_query` for questions). The `embed_hash` folder level keeps it apart from the MiniLM index automatically (06 §2.4).
- **Corpus T (Gate 2→3 item 1).** Only after DEC-3 (files chosen, hashed and frozen with snapshot manifests). Set `RUN_CORPUS_T = True` and run your one command on Corpus T as a smoke run.

Never print key values.

In [ ]:
if RUN_GROQ or RUN_COHERE or RUN_CORPUS_T:
    pass   # your code: plug in your own C3/C5 provider functions here

## Log it

Copy into your session log: your four predictions next to the measured values (chunks, `vectors.npy` bytes, rank of page 10, bytes per trace); your `params_hash`; whether Task 7b passed; your Task 8 artifact lines; your Task 9 labels and the TOY-7 sentence; the Task 10 table, your named groups and the two run flag counts. If you ran Task 12, label every answer **one run, n = 1, not a result**. The measured comparison of versions starts in Phase 3 (v0.1).